# RAG with FAISS

This notebook loads the State of the Union text, splits it into smaller `Document` chunks, embeds those chunks, and stores them in a FAISS vector index. A retriever then finds the chunks most relevant to a question.

FAISS performs vector similarity search locally. `OpenAIEmbeddings` is used to create document and query vectors, so indexing and retrieval make OpenAI API requests and may incur usage charges.

In [1]:
import os

from dotenv import find_dotenv, load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(find_dotenv())

# Load the speech as a LangChain Document.
documents = TextLoader(
    "data/state_of_the_union.txt",
    encoding="utf-8",
).load()

# Split it into overlapping chunks while preserving document metadata.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    add_start_index=True,
)
chunks = text_splitter.split_documents(documents)

print(f"Loaded {len(documents)} document(s) and created {len(chunks)} chunks.")
print("First chunk:")
print(chunks[0].page_content[:500])

Loaded 1 document(s) and created 43 chunks.
First chunk:
Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six day


In [2]:
embeddings = OpenAIEmbeddings()

faiss_index = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
)

print(f"Indexed {faiss_index.index.ntotal} chunks in the FAISS vector store.")

Indexed 43 chunks in the FAISS vector store.


In [ ]:
question = "What did the speech say about John Lewis and voting rights?"

retriever = faiss_index.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)
relevant_chunks = retriever.invoke(question)

print(f"Retrieved {len(relevant_chunks)} relevant chunks:")


for rank, document in enumerate(relevant_chunks, start=1):
    print(f"\nChunk {rank} (source: {document.metadata.get('source')}, start: {document.metadata.get('start_index')}):")
    print(document.page_content)

Retrieved 3 relevant chunks:

Chunk 1 (source: data/state_of_the_union.txt, start: 28861):
We cannot let this happen. 

Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. 

Tonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. 

One of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. 

And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.

Chunk 2 (source: data/state_of_the_union.txt, start: 30598):
We can do all this while keepi

## RAG with LCEL

LangChain Expression Language (LCEL) composes runnable steps with the `|` operator. A **Runnable** is a component with a standard input/output interface, so one step's output can be passed to the next.

`RunnablePassthrough()` returns its input unchanged. In this chain, the input is the question string, so it preserves that string for the prompt's `{question}` field. The dictionary at the start runs two branches with that same input: `retriever | RunnableLambda(format_docs)` finds relevant documents and formats them as `{context}`, while `RunnablePassthrough()` supplies `{question}`. Together, they produce a dictionary like `{"question": "...", "context": "..."}` for `rag_prompt`.

The rest of the chain formats the prompt, sends it to the chat model, and converts the model's message into plain text. Invoking it performs an embedding lookup and a model request, which may incur API usage charges.

In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI

# An LCEL Runnable is a component with a standard input/output interface.
# The | operator sends one Runnable's output to the next Runnable.

question = "What did the speech say about John Lewis and voting rights?"
def format_docs(documents):
    # The retriever returns Document objects; join their text for the prompt.
    return "\n\n".join(document.page_content for document in documents)


# This prompt expects a dictionary containing both "question" and "context".
rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer using only the provided context. If it does not contain the answer, say you do not know.",
        ),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)

# The input to rag_chain.invoke(...) is the question string.
# This mapping creates two values from that same string:
rag_chain = (
    
    {
        # Retriever: question -> relevant Documents.
        # RunnableLambda adapts our Python function into an LCEL Runnable:
        # Documents -> one context string.
        "context": retriever | RunnableLambda(format_docs),
        # Pass the original question through unchanged as the "question" value.
        "question": RunnablePassthrough(),
    }


    # The mapping produces {"context": ..., "question": ...}; fill the prompt.
    | rag_prompt
    # Send the formatted chat messages to the model; it returns an AIMessage.
    | ChatOpenAI(model="gpt-4.1-mini")
    # Extract just the response text from the AIMessage.
    | StrOutputParser()
)

# invoke runs the full pipeline: retrieve -> format -> prompt -> model -> text.
rag_answer = rag_chain.invoke(question)
print(rag_answer)

The speech called on the Senate to pass the John Lewis Voting Rights Act as part of efforts to protect voting rights, emphasizing that the right to vote and have it counted is the most fundamental right in America and is currently under assault.


# Memory in LangChain

A chat model does not automatically remember earlier turns. **Memory** is the application state that carries relevant conversation history or facts from one turn to the next. The application includes that state in later prompts and updates it with each new user message and response.

## Common Patterns

- **Buffer memory:** keeps the full conversation. It is simple, but the prompt grows with every turn and can eventually exceed the model's context limit.
- **Window memory:** keeps only a recent number of messages or turns. This bounds prompt size, but older details may be forgotten.
- **Summary memory:** compresses older turns into a summary and keeps recent turns verbatim. It saves space, but summaries can omit details.
- **Persistent or user-profile memory:** stores selected facts outside the current prompt so they can be reused later. It needs explicit user/session separation and privacy controls.

## Memory vs. RAG

Memory stores conversational state, such as what the user has already asked or prefers. The FAISS index in this notebook stores external reference material from the State of the Union speech. A RAG application can use both: retrieve document chunks for factual context and include relevant conversation history so follow-up questions make sense.

## Practical Considerations

Keep each conversation isolated by a session or thread ID; do not share one user's history with another. Decide what should be retained, where it is stored, and how it can be cleared. In-memory history is lost when the process ends unless it is backed by persistent storage.

Older LangChain tutorials often use standalone `ConversationBufferMemory`-style classes. Memory APIs vary by LangChain version; LCEL workflows can use message-history wrappers, while LangGraph applications commonly persist thread state with a checkpointer.